# CUM Series 7 — Huber SV Mapping Sweep

Deep Research V2 finding: the optimal SV mapping is `f(σ) = min(σ^α, c)` — a Huber-like function that's monotone, 2-param, and theoretically grounded.

**Scaled up from v1 benchmark:** bigger model, bigger batch, TF32 enabled.

In [ ]:
# ── Setup: clone repo, verify GPU, enable CUDA optimizations ──
import os, subprocess, sys

REPO_URL = 'https://github.com/BrownHujay/Curvature-Unified-Muon.git'
REPO_DIR = '/content/Curvature-Unified-Muon'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print(f'Cloned repo')
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    print(f'Pulled latest')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

import torch
import torch.nn as nn
import torch.nn.functional as F
import time, math
import numpy as np

# CUDA optimizations
torch.set_float32_matmul_precision('high')  # enable TF32
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

assert torch.cuda.is_available(), 'Need GPU!'
DEVICE = torch.device('cuda')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
print(f'TF32 enabled: {torch.backends.cuda.matmul.allow_tf32}')

In [ ]:
# ── Download data ──
DATA_DIR = 'benchmarks/tier0/data'
DATA_PATH = os.path.join(DATA_DIR, 'tinyshakespeare.txt')
if not os.path.exists(DATA_PATH):
    os.makedirs(DATA_DIR, exist_ok=True)
    import urllib.request
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        DATA_PATH)
with open(DATA_PATH, 'r') as f:
    TEXT = f.read()
print(f'Dataset: {len(TEXT):,} chars')

In [ ]:
# ── Imports ──
from benchmarks.models.micro_gpt import MicroGPT
from cum.newton_schulz import newton_schulz_orthogonalize
from cum.utils import aspect_ratio_scale
from cum import CUM5v6, CUM7v1
print('Imports OK')

In [ ]:
# ── Config ──
# Scaled up from v1: bigger model, bigger batch
# MicroGPT with d_model=256 gives ~6M params with meaningful matrix sizes (256x768, 256x1024)

MODEL_CFG = dict(
    d_model=256,
    n_heads=8,
    n_layers=6,
    d_ff=1024,
    ctx_len=256,
)
BATCH_SIZE = 128
TOTAL_STEPS = 2000
WARMUP_STEPS = 200
EVAL_EVERY = 250  # more frequent evals for better trajectory
BASE_LR = 0.02
SEED = 42

# Quick check model size
test_model = MicroGPT(vocab_size=65, **MODEL_CFG)
n_params = sum(p.numel() for p in test_model.parameters())
n_hidden = sum(p.numel() for p in test_model.get_hidden_2d_params())
hidden_shapes = [(p.shape[0], p.shape[1]) for p in test_model.get_hidden_2d_params()]
print(f'Model: {n_params/1e6:.1f}M params ({n_hidden/1e6:.1f}M hidden 2D)')
print(f'Hidden matrix shapes: {set(hidden_shapes)}')
print(f'Batch size: {BATCH_SIZE}')
del test_model

In [ ]:
# ── Data & Training Infrastructure ──

class CharDataset:
    def __init__(self, text, ctx_len=256, device='cpu'):
        self.chars = sorted(set(text))
        self.char_to_idx = {c: i for i, c in enumerate(self.chars)}
        self.vocab_size = len(self.chars)
        self.data = torch.tensor([self.char_to_idx[c] for c in text], dtype=torch.long, device=device)
        self.ctx_len = ctx_len

    def get_batch(self, batch_size, split='train', train_ratio=0.9):
        n = int(len(self.data) * train_ratio)
        data = self.data[:n] if split == 'train' else self.data[n:]
        ix = torch.randint(0, len(data) - self.ctx_len - 1, (batch_size,), device=self.data.device)
        x = torch.stack([data[i:i+self.ctx_len] for i in ix])
        y = torch.stack([data[i+1:i+self.ctx_len+1] for i in ix])
        return x, y


class SimpleMuon(torch.optim.Optimizer):
    def __init__(self, params, lr=0.02, beta1=0.95, ns_steps=5):
        defaults = dict(lr=lr, beta1=beta1, ns_steps=ns_steps)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                state = self.state[p]
                if len(state) == 0:
                    state['momentum_buffer'] = torch.zeros_like(p)
                m = state['momentum_buffer']
                m.mul_(group['beta1']).add_(p.grad, alpha=1 - group['beta1'])
                u = p.grad + group['beta1'] * m
                orth = newton_schulz_orthogonalize(u, steps=group['ns_steps'])
                scale = math.sqrt(max(1, p.shape[0] / p.shape[1]))
                p.add_(orth, alpha=-group['lr'] * scale)


def get_lr(step, total_steps, warmup_steps, base_lr):
    if step < warmup_steps:
        return base_lr * step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return base_lr * 0.5 * (1 + math.cos(math.pi * progress))


dataset = CharDataset(TEXT, ctx_len=MODEL_CFG['ctx_len'], device=DEVICE)
print(f'Dataset on {DEVICE}: vocab={dataset.vocab_size}')

In [ ]:
# ── Training Loop ──

def train_one(name, main_opt, aux_opt, model, dataset,
              total_steps=TOTAL_STEPS, batch_size=BATCH_SIZE,
              warmup_steps=WARMUP_STEPS, base_lr=BASE_LR, eval_every=EVAL_EVERY):
    model.train()
    trajectory = []
    t0 = time.perf_counter()

    # Warmup torch.compile
    for _ in range(3):
        x, y = dataset.get_batch(batch_size, split='train')
        _, loss = model(x, y)
        loss.backward()
        if main_opt: main_opt.zero_grad()
        aux_opt.zero_grad()

    torch.cuda.synchronize()
    t0 = time.perf_counter()  # reset after warmup

    for step in range(total_steps):
        current_lr = get_lr(step, total_steps, warmup_steps, base_lr)
        if main_opt:
            for g in main_opt.param_groups:
                g['lr'] = current_lr

        x, y = dataset.get_batch(batch_size, split='train')
        _, loss = model(x, y)

        if main_opt: main_opt.zero_grad()
        aux_opt.zero_grad()
        loss.backward()
        if main_opt: main_opt.step()
        aux_opt.step()

        if step % eval_every == 0 or step == total_steps - 1:
            model.eval()
            vl = []
            for _ in range(20):
                vx, vy = dataset.get_batch(batch_size, split='val')
                with torch.no_grad():
                    _, v = model(vx, vy)
                    vl.append(v.item())
            val_loss = np.mean(vl)
            trajectory.append((step, val_loss))
            model.train()
            elapsed = time.perf_counter() - t0
            print(f'  [{name}] step {step}: val_loss={val_loss:.4f} ({elapsed:.0f}s)')

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    # Final eval (more samples)
    model.eval()
    vl = []
    for _ in range(50):
        vx, vy = dataset.get_batch(batch_size, split='val')
        with torch.no_grad():
            _, v = model(vx, vy)
            vl.append(v.item())
    final_val = np.mean(vl)
    return final_val, trajectory, elapsed


print('Training loop ready')

In [ ]:
# ── Optimizer Factory ──

def make_model_and_opts(dataset, cfg):
    torch.manual_seed(SEED)
    model = MicroGPT(vocab_size=dataset.vocab_size, **MODEL_CFG).to(DEVICE)
    model = torch.compile(model, mode='reduce-overhead')

    hidden_2d = model.get_hidden_2d_params()
    other = model.get_other_params()
    aux_opt = torch.optim.AdamW(other, lr=3e-4, weight_decay=0.01)

    t = cfg['type']

    if t == 'Muon':
        main_opt = SimpleMuon(hidden_2d, lr=BASE_LR, ns_steps=5)
    elif t == '5v6':
        main_opt = CUM5v6(
            hidden_2d, lr=BASE_LR,
            mode=cfg.get('mode', 'ns_blend'),
            ns_save_at=cfg.get('ns_save_at', 2),
            ns_blend=cfg.get('ns_blend', 0.25),
        )
    elif t == '7v1':
        main_opt = CUM7v1(
            hidden_2d, lr=BASE_LR,
            mode=cfg.get('mode', 'huber'),
            alpha=cfg.get('alpha', 0.3),
            cap=cfg.get('cap', 0.88),
            alpha_start=cfg.get('alpha_start', 0.5),
            alpha_end=cfg.get('alpha_end', 0.1),
            total_steps=TOTAL_STEPS,
            threshold=cfg.get('threshold', 0.1),
        )
    else:
        raise ValueError(f'Unknown: {t}')

    return model, main_opt, aux_opt


def run_all(configs):
    results = []
    for i, cfg in enumerate(configs):
        name = cfg['name']
        print(f'\n{"="*60}')
        print(f'[{i+1}/{len(configs)}] {name}')
        print(f'{"="*60}')
        try:
            model, main_opt, aux_opt = make_model_and_opts(dataset, cfg)
            val_loss, traj, elapsed = train_one(name, main_opt, aux_opt, model, dataset)
            results.append(dict(name=name, type=cfg['type'], val_loss=val_loss,
                                trajectory=traj, elapsed=elapsed, error=None))
            print(f'  FINAL: {name} -> {val_loss:.4f} ({elapsed:.1f}s)')
        except Exception as e:
            import traceback; traceback.print_exc()
            results.append(dict(name=name, type=cfg['type'], val_loss=float('inf'),
                                trajectory=[], elapsed=0, error=str(e)))
        torch.cuda.empty_cache()
    return results


print('Factory ready')

## Series 7a: Huber Sweep

`f(σ) = min(σ^α, c)` — sweep α (equalization strength) × c (sub-unity cap)

In [ ]:
CONFIGS_7A = [
    # Baselines
    {'name': 'Muon NS=5',           'type': 'Muon'},
    {'name': '5v6 best (s2 b=0.25)', 'type': '5v6', 'ns_save_at': 2, 'ns_blend': 0.25},

    # Huber: sweep alpha (equalization) x cap (contraction)
    # α=0.1 = aggressive equalization, α=0.5 = moderate, α=0.7 = mild
    # c=0.85/0.88/0.92 = different contraction strengths
    {'name': 'Huber a=0.1 c=0.88', 'type': '7v1', 'mode': 'huber', 'alpha': 0.1, 'cap': 0.88},
    {'name': 'Huber a=0.3 c=0.85', 'type': '7v1', 'mode': 'huber', 'alpha': 0.3, 'cap': 0.85},
    {'name': 'Huber a=0.3 c=0.88', 'type': '7v1', 'mode': 'huber', 'alpha': 0.3, 'cap': 0.88},
    {'name': 'Huber a=0.3 c=0.92', 'type': '7v1', 'mode': 'huber', 'alpha': 0.3, 'cap': 0.92},
    {'name': 'Huber a=0.5 c=0.88', 'type': '7v1', 'mode': 'huber', 'alpha': 0.5, 'cap': 0.88},
    {'name': 'Huber a=0.7 c=0.88', 'type': '7v1', 'mode': 'huber', 'alpha': 0.7, 'cap': 0.88},

    # Smooth Huber: c * tanh(σ^α / c)
    {'name': 'SmoothHub a=0.3 c=0.88', 'type': '7v1', 'mode': 'huber_smooth', 'alpha': 0.3, 'cap': 0.88},

    # Pure power: Schatten-p descent at cap
    {'name': 'Power a=0.3 c=0.88',  'type': '7v1', 'mode': 'power', 'alpha': 0.3, 'cap': 0.88},

    # Scheduled: α anneals from 0.5 (less EQ early) to 0.1 (more EQ late)
    {'name': 'Sched 0.5->0.1 c=0.88', 'type': '7v1', 'mode': 'scheduled_power', 'alpha_start': 0.5, 'alpha_end': 0.1, 'cap': 0.88},
    {'name': 'Sched 0.3->0.05 c=0.88', 'type': '7v1', 'mode': 'scheduled_power', 'alpha_start': 0.3, 'alpha_end': 0.05, 'cap': 0.88},
]

print(f'{len(CONFIGS_7A)} configs to run')

In [ ]:
# RUN
results_7a = run_all(CONFIGS_7A)

In [ ]:
# ── Results ──
def show_results(results, series='7a'):
    muon = next((r['val_loss'] for r in results if r['type'] == 'Muon'), None)
    v56 = next((r['val_loss'] for r in results if r['type'] == '5v6'), None)

    print(f'## Series {series} Results\n')
    print(f'| Config | Val Loss | vs Muon | vs 5v6 | Time |')
    print(f'|--------|----------|---------|--------|------|')
    for r in sorted(results, key=lambda x: x['val_loss']):
        if r['error']:
            print(f'| {r["name"]} | FAILED | — | — | {r["error"][:30]} |')
            continue
        vm = f'{r["val_loss"] - muon:+.4f}' if muon else '—'
        v5 = f'{r["val_loss"] - v56:+.4f}' if v56 else '—'
        print(f'| {r["name"]} | {r["val_loss"]:.4f} | {vm} | {v5} | {r["elapsed"]:.0f}s |')

    new = [r for r in sorted(results, key=lambda x: x['val_loss'])
           if r['type'] not in ('Muon', '5v6') and not r['error']]
    if new:
        b = new[0]
        print(f'\nBEST NEW: {b["name"]} -> {b["val_loss"]:.4f}', end='')
        if muon: print(f' (vs Muon: {b["val_loss"]-muon:+.4f})', end='')
        if v56: print(f' (vs 5v6: {b["val_loss"]-v56:+.4f})', end='')
        print()

show_results(results_7a)

## Series 7b: Refine best Huber params

Narrow sweep around the best α and c from 7a.

In [ ]:
# TODO: fill in based on 7a results
CONFIGS_7B = [
    {'name': 'Muon NS=5',           'type': 'Muon'},
    {'name': '5v6 best (s2 b=0.25)', 'type': '5v6', 'ns_save_at': 2, 'ns_blend': 0.25},
    # Add narrow sweep around best from 7a
]

if len(CONFIGS_7B) > 2:
    results_7b = run_all(CONFIGS_7B)
    show_results(results_7b, '7b')
else:
    print('Fill in configs based on 7a, then re-run')